# **Automatic Access Reviewer**

# **Preparing and installing all the requirement to run the Automatic Access Checker**


In [ ]:
# Install necessary packages
!pip install pandas
!pip install numpy
!pip install requests
!pip install beautifulsoup4
!pip install xlrd


In [ ]:
# Importing necessary libraries as the first step
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re

Code initialization and data reading from each of the excel, spreadsheet, and txt file.

In [ ]:
# reading data from user data in html format.

# Specify the path to your HTML file in Google Drive
file_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\User data Created at_ May 26, 2025, 3_10_17 PM.html'

# Read the HTML file into a Pandas DataFrame
with open(file_path, 'r', encoding='utf-8') as file:
    soup = BeautifulSoup(file, 'html.parser')
tables = pd.read_html(str(soup))

# Convert the first table to a DataFrame and display the first few rows
tables_df = pd.DataFrame(tables[0])
tables_df.head()

pattern = r'[•]'

# Ensure all values in tables_df[4] are strings
tables_df[1] = tables_df[1].astype(str)

# Use regex substitution to remove special characters from the 'text' column
tables_df[1] = tables_df[1].apply(lambda x: re.sub(pattern, ';', x))

pattern = r'[•]'

# Ensure all values in tables_df[4] are strings
tables_df[3] = tables_df[3].astype(str)

# Ensure all values in tables_df[4] are strings
tables_df[4] = tables_df[4].astype(str)

# Use regex substitution to remove special characters from the 'text' column
tables_df[3] = tables_df[3].apply(lambda x: re.sub(pattern, ';', x))

# Use regex substitution to remove special characters from the 'text' column
tables_df[4] = tables_df[4].apply(lambda x: re.sub(pattern, ';', x))

# Print the updated DataFrame
print(tables_df[3])

# Print the updated DataFrame
print(tables_df[4])

tables_df.head()

# The KMS User data will be stored into csv file.
KMS_User_data_raw_extraction = tables_df.to_csv(r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_user_data.csv', index=False)

def extract_user_info(user_info_str):
    """Extract user information from the formatted string."""
    # Check if input is valid
    if pd.isna(user_info_str) or not isinstance(user_info_str, str):
        print(f"Invalid user info input: {user_info_str}")
        return {
            'First Name': '',
            'Last Name': '',
            'E-mail address': '',
            'Mobile phone number': '',
            'Department': '',
            'Terminal user level': '',
            'User Groups': ''
        }

    # Initialize dictionary with empty values
    info = {
        'First Name': '',
        'Last Name': '',
        'E-mail address': '',
        'Mobile phone number': '',
        'Department': '',
        'Terminal user level': '',
        'User Groups': ''
    }

    try:
        # Define patterns to extract information
        patterns = {
            'First Name': r'First name:\s*(.*?)(?=Last name:|$)',
            'Last Name': r'Last name:\s*(.*?)(?=E-mail address:|$)',
            'E-mail address': r'E-mail address:\s*(.*?)(?=Mobile phone number:|$)',
            'Mobile phone number': r'Mobile phone number:\s*(.*?)(?=Landline number:|$)',
            'Department': r'Department:\s*(.*?)(?=Terminal user level:|$)',
            'Terminal user level': r'Terminal user level:\s*(.*?)(?=Max|$)'
        }

        # Extract each field using regex
        for field, pattern in patterns.items():
            match = re.search(pattern, user_info_str, re.DOTALL)
            if match:
                info[field] = match.group(1).strip()

        # Extract user groups from user group section
        user_groups_pattern = r'user group:\s*(.*?)(?=Valid:|$)'
        user_groups_match = re.search(user_groups_pattern, user_info_str, re.DOTALL)
        if user_groups_match:
            groups_text = user_groups_match.group(1)
            # Extract group names (Admin or full Cust-XXX patterns)
            group_names = re.findall(r'\d+=(Admin|Cust-\d+(?:-[A-Z]+)?)', groups_text)
            # Join the group names with semicolons
            info['User Groups'] = ' ;'.join(group_names) if group_names else ''

        return info

    except Exception as e:
        print(f"Error extracting user info: {e}")
        print(f"From string: {user_info_str}")
        return info

def extract_credentials(credentials_str):
    """Extract RFID from credentials string."""
    if pd.isna(credentials_str) or not isinstance(credentials_str, str):
        return {'RFID': ''}

    try:
        # Extract RFID (number after comma)
        rfid_match = re.search(r',\s*(\d+)', credentials_str)
        rfid = rfid_match.group(1) if rfid_match else ''

        return {
            'RFID': rfid
        }
    except Exception as e:
        print(f"Error processing credentials: {e}")
        return {'RFID': ''}

def process_csv(KMS_User_data_raw_extraction, output_file_path):
    """Process the input CSV file and create a new formatted CSV."""
    try:
        # Read the input CSV file
        data = pd.read_csv(KMS_User_data_raw_extraction)

        # Print column names for debugging
        print("Available columns in CSV:", data.columns.tolist())

        # Skip the header row and process from the second row
        data_processed = data.iloc[1:].copy()

        # Initialize list to store processed records
        processed_records = []

        # Process each row
        for index, row in data_processed.iterrows():
            try:
                # Skip rows that contain administrative information
                if 'Administrator' in str(row['0']):
                    continue

                # Extract information using the correct column indices
                user_info = extract_user_info(str(row['1']))
                cred_info = extract_credentials(str(row['2']))
                key_tag = str(row['3']) if not pd.isna(row['3']) else ''
                cust_group = str(row['4']) if not pd.isna(row['4']) else ''

                # Remove leading/trailing semicolons and spaces from key_tag and cust_group
                key_tag = key_tag.strip('; ')
                cust_group = cust_group.strip('; ')

                # Combine all information
                record = {
                    **user_info,
                    **cred_info,
                    'Key tag': key_tag,
                    'Cust group': cust_group
                }

                processed_records.append(record)

            except Exception as e:
                print(f"Error processing row {index}:")
                print(f"Row data: {row}")
                print(f"Error details: {e}")
                continue

        # Create DataFrame with processed records
        columns = [
            'First Name', 'Last Name', 'E-mail address', 'Mobile phone number',
            'Department', 'Terminal user level', 'User Groups', 'RFID',
            'Key tag', 'Cust group'
        ]

        output_df = pd.DataFrame(processed_records, columns=columns)

        # Print summary
        print(f"\nProcessed {len(processed_records)} records")
        print("\nFirst few rows of processed data:")
        print(output_df.head())

        # Save to CSV
        output_df.to_csv(output_file_path, index=False)
        print(f"\nProcessed data saved to {output_file_path}")
        return output_df

    except Exception as e:
        print(f"Error processing CSV file: {e}")
        return pd.DataFrame()

# Usage example:
if __name__ == "__main__":
    # Replace these with your actual file paths
    KMS_User_data_raw_extraction = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_user_data.csv'
    output_file = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_user_data_final.csv'
    processed_data = process_csv(KMS_User_data_raw_extraction, output_file)

KMS_User_data_Final_df = pd.read_csv(r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_user_data_final.csv')
KMS_Name_RFID = KMS_User_data_Final_df.iloc[:, [0, 1, 7]]


This cell will convert and pre-process all the data to be readable by the next cell of codes.

Please update the last company written on CAAR summary.
Example: Company Name |	Zenlayer, Inc. (see on each sheet: C4 cell)

In [ ]:
import pandas as pd
import re

# Load the data
LIA_df = pd.read_csv(r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\RadGridExport (3).csv')

# Define a function to extract only the numbers from the Credential field
def extract_number(credential):
    # Use regex to find sequences of digits in the credential string
    match = re.search(r'\d+', str(credential))
    return match.group() if match else None

# Apply the function to the 'Credential' column
LIA_df['Credential'] = LIA_df['Credential'].apply(extract_number)

# Select only 'Credential Owner' and 'Credential' columns
LIA_name_RFID_df = LIA_df[['Credential Owner', 'Credential']]

# Convert all columns to string type
LIA_name_RFID_df = LIA_name_RFID_df.astype(str)

# Save the resulting DataFrame to a new CSV file with text type
LIA_name_RFID_df.to_csv(r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\LIA_Name_Credentials.csv', index=False)

# Display the resulting DataFrame
print(LIA_name_RFID_df.head())

In [ ]:
#Importing data from P1

P1_DF = pd.read_excel(r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\Staff 260525.xls", engine='xlrd')

P1_staff_name_df = P1_DF.iloc[:,1]
P1_user_data_RFID = P1_DF.iloc[:, [1, 5]]
P1_staff_name_df.head()
P1_staff_name_df.to_csv(r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\P1_Staff_Name.csv")

In [ ]:
import pandas as pd

# Path to your local Excel file
excel_file_path = r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG1 CAAR Summary (1).xlsx"

# Prompt for company name
last_company = input("Masukkan nama company terakhir (dapat dicopy di CAAR Summary cell C4, contoh 'PT. Fiber Networks Indonesia') = ")

# Initialization
df_caar_summary_EG1 = pd.DataFrame()
length_cust = []
cust_name = []
cust_list_id = []
looper = False

# Load all sheets
xls = pd.ExcelFile(excel_file_path)
sheet_names = xls.sheet_names

print("Taking data from CAAR Summary to be used as source data")

# Loop through all sheets
for sheet_name in sheet_names:
    df_sheet = pd.read_excel(excel_file_path, sheet_name=sheet_name, header=None)

    # Check if it has the 'Back to Summary' marker
    if df_sheet.shape[1] > 1 and 'Back to Summary' in str(df_sheet.iloc[0, 1]):
        current_company = str(df_sheet.iloc[3, 2])

        # Check for mismatched format
        if df_sheet.iloc[12:].shape[1] != 14:
            print(f"{current_company} has different CAAR Summary Format, please fix the format first before continue running the program")
            break

        print("Concatenating", current_company)

        # Append CAAR data
        df_caar_summary_EG1 = pd.concat([df_caar_summary_EG1, df_sheet.iloc[12:]], axis=0, ignore_index=True)
        length_cust.append(len(df_caar_summary_EG1))
        cust_name.append(current_company)

        temp_id = str(df_sheet.iloc[5, 2]).rstrip(" ")
        temp_id = temp_id[-3:]
        print(temp_id)
        cust_list_id.append(temp_id)

        # If this is the last company, break
        if last_company in current_company:
            looper = True
            break

if looper:
    # Rename columns and clean up
    df_result_EG1 = df_caar_summary_EG1.rename(columns={
        0: "Index", 1: "Name", 2: "Authorization", 3: "NIK", 4: "Access Number",
        5: "Status", 6: "Phone Number", 7: "E-mail", 8: "pin", 9: "Kartu",
        10: "Foto", 11: "Finger & PIN", 12: "Face Recog"
    })

    df_result_EG1 = df_result_EG1.dropna(subset=['Name'])

    print("Finished pulling data from CAAR summary, here is a glimpse of the result:")
    print(df_result_EG1.head(5))

    # Export to CSV
    output_path = r"C:\Users\adonis.hugo\Downloads\Automatic accress revew\CAAR_Summary_EG1.csv"
    df_result_EG1.to_csv(output_path, index=False)
    print(f"Saved to {output_path}")
else:
    print("Specified company not found or loop stopped early.")


In [ ]:
import pandas as pd

# Path to your local Excel file
excel_file_path = r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG2 New CAAR Summary (6).xlsx"

# User input
last_company = input("Masukkan nama company terakhir (dapat dicopy di CAAR Summary cell C4, contoh 'PT. Fiber Networks Indonesia') = ")

# Initialization
count = 0
looper = 0
df_caar_summary_EG2 = pd.DataFrame()
length_cust = []
cust_name = []
cust_list_id = []

# Load Excel file
xls = pd.ExcelFile(excel_file_path)
sheet_names = xls.sheet_names

print("Taking data from CAAR Summary to be used as source data")

# Loop through sheets
for sheet_name in sheet_names:
    try:
        from_caar_summary = pd.read_excel(xls, sheet_name=sheet_name, header=None)

        marker_found = False
        if 'To complete this form, please fill in the column marked with an asterisk (*)' in str(from_caar_summary.iloc[0, 1]):
            marker_found = True
        elif 'Back to Summary' in str(from_caar_summary.iloc[0, 1]):
            marker_found = True
        elif len(from_caar_summary) > 8:
            if any('To complete this form' in str(from_caar_summary.iloc[7, col]) for col in range(min(3, from_caar_summary.shape[1]))):
                marker_found = True

        if marker_found:
            if 'Back to Summary' in str(from_caar_summary.iloc[0, 1]):
                # Old format
                company_name = from_caar_summary.iloc[11, 3]
                customer_id_row = 5
                customer_id_col = 2
                data_start_row = 12
            else:
                # New format
                company_name = from_caar_summary.iloc[11, 3]
                customer_id_row = 12
                customer_id_col = 3
                data_start_row = 12

            if last_company in str(company_name):
                looper = 1

            if from_caar_summary.iloc[data_start_row:].shape[1] < 13:
                print(f"{company_name} has different CAAR Summary Format, please fix the format first before continue running the program")
                continue

            print("Concatenating", company_name)

            # Append data
            df_caar_summary_EG2 = pd.concat([df_caar_summary_EG2, from_caar_summary.iloc[data_start_row:]], axis=0, ignore_index=True)
            length_cust.append(len(df_caar_summary_EG2))
            cust_name.append(str(company_name))

            # Extract customer ID
            try:
                temp_id = str(from_caar_summary.iloc[customer_id_row, customer_id_col])
                if "CUS-" in temp_id:
                    temp_id = temp_id.split("-")[1]
                else:
                    temp_id = temp_id.rstrip(" ")
                    temp_id = temp_id[-3:]
                print(temp_id)
                cust_list_id.append(temp_id)
            except:
                print("Could not extract customer ID")
                cust_list_id.append("000")

        if looper == 1:
            break

    except Exception as e:
        print(f"Error at sheet '{sheet_name}': {str(e)}")
        looper = 1

# Format result
df_result_EG2 = df_caar_summary_EG2.rename(columns={
    0: "Index",
    1: "Name",
    2: "Authorization",
    3: "NIK",
    4: "Access Number",
    5: "Status",
    6: "Phone Number",
    7: "E-mail",
    8: "pin",
    9: "Kartu",
    10: "Foto",
    11: "Finger & PIN",
    12: "Face Recog",
    13: "13"  # placeholder to match EG1
})

# Clean and filter rows
df_result_EG2 = df_result_EG2.dropna(subset=['Name'])
df_result_EG2 = df_result_EG2[df_result_EG2['Name'].astype(str).str.strip() != '']
df_result_EG2.insert(0, "", range(1, len(df_result_EG2) + 1))

# Show and save
print("Finished pulling data from CAAR summary, here is a glimpse of the result:")
print(df_result_EG2.head(5))

# Export
output_path = r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG2_Summary.csv"
df_result_EG2.to_csv(output_path, index=False)
print(f"Saved to {output_path}")


# **Checking Users' status on both P1 (For EDGE 1) and LIA (For EDGE 2)**

In [ ]:
# Working on this one.

import pandas as pd

# Load the CSV
df_result_EG1_relevant = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\CAAR_Summary_EG1.csv'
df_result_EG1_Relevant_test_df = pd.read_csv(df_result_EG1_relevant)

# Extract columns 4 and 13
df_result_EG1_Relevant_use_df = df_result_EG1_Relevant_test_df.iloc[:, [1, 5]]

# Save to new CSV
df_result_EG1_Relevant_use_df.to_csv(
    r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\CAAR_EG1_Relevant.csv", index=False)

# Show the first 100 rows
print(df_result_EG1_Relevant_test_df.head(100))


In [ ]:
import pandas as pd
import difflib

# Path to the CSV file
CAAR_EG1_relevant_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\CAAR_EG1_Relevant.csv'

# STEP 1: Load the CSV while skipping the first redundant header row
raw_df = pd.read_csv(CAAR_EG1_relevant_path, header=None, skiprows=1)

# STEP 2: Use the first row (now index 0) as the actual header
header_row_index = 0
header = raw_df.iloc[header_row_index].fillna('').astype(str).str.strip()
data = raw_df.iloc[header_row_index + 1:].copy()
data.columns = header
data.columns = data.columns.str.strip()
data.reset_index(drop=True, inplace=True)

# STEP 3: Define expected access-related columns
expected_columns = ['Status', 'Kartu', 'Foto', 'Finger & PIN', 'Face Recog', 'KMS']

# STEP 4: Fuzzy match expected columns to actual
actual_columns = list(data.columns)
available_columns = []
column_map = {}

for col in expected_columns:
    match = difflib.get_close_matches(col, actual_columns, n=1, cutoff=0.6)
    if match:
        matched_col = match[0]
        available_columns.append(matched_col)
        column_map[col] = matched_col

# Show matches
print("\nExpected to actual column match:")
if column_map:
    for expected, actual in column_map.items():
        print(f"  '{expected}' -> '{actual}'")
else:
    print("⚠️ No access-related columns matched. Proceeding with only Name if found.")

# STEP 5: Filter out rows where all matched access columns are empty (if any)
if available_columns:
    filtered_df = data.dropna(subset=available_columns, how='all')
else:
    filtered_df = data.copy()

# STEP 6: Fuzzy match for 'Name' column
possible_name_cols = difflib.get_close_matches('Name', actual_columns, n=1, cutoff=0.6)
if not possible_name_cols:
    raise KeyError("Could not find a column similar to 'Name' in the dataset.")
name_col = possible_name_cols[0]

# STEP 7: Remove duplicates based on Name
final_df = filtered_df.drop_duplicates(subset=name_col, keep='first')

# STEP 8: Prepare final output columns
output_columns = [name_col] + available_columns
output_df = final_df[output_columns].rename(columns={name_col: 'Name'} | column_map)
output_df.reset_index(drop=True, inplace=True)

# STEP 9: Save to CSV
output_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\filtered_output_EG1.csv'
output_df.to_csv(output_path, index=False)

# STEP 10: Show preview
print("\nFiltered Data Preview:")
print(output_df.head())
print(f"\n✅ Total filtered names: {len(output_df)}")
print(f"📁 Output saved to: {output_path}")

In [ ]:
# P1 access checker with better heading.
import pandas as pd

# File paths
filtered_output_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\filtered_output_EG1.csv'  # Replace with your actual path
staff_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\Staff 260525.xls'  # Replace with your actual path
output_csv_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\P1_Credential_Checked.csv'  # Output file path

try:
    # Read the filtered_output_EG1.csv with a specified encoding
    filtered_output_df = pd.read_csv(filtered_output_path)

    # Read the Staff 170424.xls using xlrd engine for .xls files
    staff_df = pd.read_excel(staff_path, engine='xlrd')

    # Ensure column names are standardized for comparison
    filtered_output_df.columns = filtered_output_df.columns.str.strip().str.lower()
    staff_df.columns = staff_df.columns.str.strip().str.lower()

    # Ensure 'name' columns are strings
    filtered_output_df['name'] = filtered_output_df['name'].astype(str)
    staff_df['staff name'] = staff_df['staff name'].astype(str)
    staff_data = staff_df[['staff name', 'card no']].rename(columns={'staff name': 'name'})

    # Initialize lists for different result categories
    active_messages = []
    terminated_messages = []
    pending_messages = []
    not_in_p1_messages = []  # Renamed from "No Access"
    not_a_customer_messages = []  # Renamed from "Doesn't exist in EDGE 1"

    # Iterate over the names in the filtered_output_EG1.csv
    for _, row in filtered_output_df.iterrows():
        name, status = str(row['name']), row['status']

        # Check if the name exists in the Staff 170424.xls data
        staff_record = staff_data[staff_data['name'].str.lower() == name.lower()]

        if not staff_record.empty:
            card_no = staff_record['card no'].values[0]

            if pd.notna(card_no):  # Check if card number is present
                if str(status).lower() == 'active':
                    active_messages.append(f'"{name}" is still active and has access in P1')
                elif str(status).lower() == 'terminated':
                    terminated_messages.append(f'"{name}" has been terminated, but still have access in P1. Please remove from P1')
                elif str(status).lower() == 'pending':
                    pending_messages.append(f'"{name}"\'s status is still pending, but exists in CAAR EG1. Please remove from P1')
            else:  # Card number is missing
                if str(status).lower() == 'active':
                    active_messages.append(f'"{name}" is active in CAAR EG1, but doesnt have access in P1')
        else:  # Name not found in Staff 170424.xls
            not_in_p1_messages.append(f'"{name}" exists in CAAR EG1, but not in P1')

    # Check for names in Staff 170424.xls not present in filtered_output_EG1.csv
    for _, row in staff_data.iterrows():
        name = str(row['name'])
        if name.lower() not in filtered_output_df['name'].str.lower().values:
            not_a_customer_messages.append(f'"{name}" doesn\'t exist in CAAR EDGE 1, but has exist in P1. Not a customer')

    # Create a DataFrame with the results
    results_df = pd.DataFrame({
        'Active': pd.Series(active_messages),
        'Terminated': pd.Series(terminated_messages),
        'Pending': pd.Series(pending_messages),
        'Not in P1': pd.Series(not_in_p1_messages),  # Renamed column
        'Not a customer': pd.Series(not_a_customer_messages)  # Renamed column
    })

    # Save results to a CSV file
    results_df.to_csv(output_csv_path, index=False)
    print(f"Results have been saved to {output_csv_path}")

except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
# Currently, working on this (Most recent one)
import pandas as pd

# Load the CSV
df_result_EG2_Relevant_test = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG2_Summary.csv'
df_result_EG2_Relevant_test_df = pd.read_csv(df_result_EG2_Relevant_test)
df_result_EG2_Relevant_df = df_result_EG2_Relevant_test_df.iloc[:, [4, 13]]
df_result_EG2_Relevant_df.to_csv(r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG2_Summary_Relevant.csv')
# Show the first 100 rows
print(df_result_EG2_Relevant_df.head(100))


In [ ]:
import pandas as pd
import re
import os

def filter_customer_data(input_file, output_file):
    """
    Extract customer names and their registration statuses from the CSV file,
    filtering out everything else.

    Args:
        input_file (str): Path to the input CSV file
        output_file (str): Path to save the filtered data
    """
    # Read the CSV file without headers
    df = pd.read_csv(input_file, header=None)

    # Rename columns for easier processing
    df.columns = ['Index', 'Column1', 'Column2'] if len(df.columns) >= 3 else df.columns

    # Find rows where the headers appear (start of sections)
    header_rows = df[df['Column1'] == 'CUSTOMER NAME*'].index.tolist()

    # Initialize list to store customer data
    customer_data = []

    # Process each section
    for i in range(len(header_rows)):
        start_idx = header_rows[i] + 1  # Start from the row after the header

        # Determine end index (either the next header or the end of the dataframe)
        end_idx = header_rows[i+1] if i < len(header_rows) - 1 else len(df)

        # Extract section data
        section = df.iloc[start_idx:end_idx].copy()

        # Process each row in the section
        for _, row in section.iterrows():
            name = row['Column1']

            # Skip non-name entries
            if not is_likely_person_name(name):
                continue

            # Convert NaN values to empty strings to avoid issues
            status = row['Column2'] if pd.notna(row['Column2']) else ""

            # Transform status values
            if status.lower() == "completed":
                status = "Active"
            elif status.lower() == "partial":
                status = "Pending"
            # "Terminated" remains unchanged

            customer_data.append({
                'Name': name,
                'Status': status
            })

    # Create DataFrame from collected data
    result_df = pd.DataFrame(customer_data)

    # Save to CSV
    result_df.to_csv(output_file, index=False)

    print(f"✅ Processed {len(result_df)} customer records.")
    print(f"💾 Filtered data saved to: {os.path.abspath(output_file)}")

    return result_df

def is_likely_person_name(value):
    """
    Determines if a value is likely to be a person's name.

    Args:
        value: The value to check

    Returns:
        bool: True if the value is likely a person name, False otherwise
    """
    if pd.isna(value) or value == "":
        return False
    if not isinstance(value, str):
        return False

    name_str = str(value).strip()

    if re.search(r'^\d+$', name_str): return False
    if re.search(r'^\d{5,}$', name_str): return False
    if re.search(r'^\+?\d[\d\s-]+$', name_str): return False
    if re.search(r'^Level\s+\d+$', name_str, re.IGNORECASE): return False
    if re.search(r'^\d+\s?[kK][vV][aA]', name_str): return False
    if re.search(r'^\d+\s?(?:rack|unit|u)$', name_str, re.IGNORECASE): return False
    if re.search(r'^CUS-\d+$', name_str): return False
    if re.search(r'^[A-Z]{3,}ID$', name_str): return False
    if re.search(r'ID', name_str, re.IGNORECASE): return False
    if re.search(r'^Government', name_str, re.IGNORECASE): return False
    if re.search(r'^KTP', name_str, re.IGNORECASE): return False
    if re.search(r'^PASS', name_str, re.IGNORECASE): return False
    if re.search(r'^Passport', name_str, re.IGNORECASE): return False
    if re.search(r'^SIMC', name_str, re.IGNORECASE): return False
    if re.search(r'/\s*[A-Z0-9]', name_str, re.IGNORECASE): return False
    if re.search(r'\d{4}[-\s]?\d{4}', name_str): return False
    if re.search(r'^only|^\(only', name_str): return False
    if re.search(r'^Photo\s+', name_str): return False
    if '@' in name_str: return False
    if '/' in name_str: return False
    if re.search(r'\d{4,}', name_str): return False

    words = name_str.split()

    if len(name_str) <= 2:
        return False
    if name_str.isupper() and len(words) == 1:
        return False
    if len(words) >= 2 and len(name_str) > 3:
        return True
    if len(words) == 1 and len(name_str) >= 4:
        if name_str[0].isupper() and not name_str.isupper():
            return True

    return False

if __name__ == "__main__":
    # 🔧 Set these paths to your local files
    input_file = r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG2_Summary_Relevant.csv"
    output_file = r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG2_CAAR_Filtered.csv"

    if not os.path.exists(input_file):
        print("❌ Input file does not exist. Please check the path and try again.")
    else:
        filtered_data = filter_customer_data(input_file, output_file)

        print("\n📋 Sample of filtered data:")
        print(filtered_data.head(10))

        print("\n📊 Registration status counts:")
        status_counts = filtered_data['Status'].value_counts()
        print(status_counts)


In [ ]:
# File paths - change these to match the actual paths on your local machine
radgrid_file = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\RadGridExport (3).csv'
filtered_output_file = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG2_CAAR_Filtered.csv'
output_csv_file = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\LIA_Credentials_Checked.csv'

# Load the CSV files
import pandas as pd

radgrid_df = pd.read_csv(radgrid_file)
filtered_output_df = pd.read_csv(filtered_output_file)

# Clean and preprocess the data
filtered_output_df['Name'] = filtered_output_df['Name'].fillna('').astype(str)
filtered_output_df['Status'] = filtered_output_df['Status'].fillna('').astype(str)

# Extract relevant columns and normalize
radgrid_names = radgrid_df['Credential Owner'].dropna().str.lower().str.strip()
filtered_names = filtered_output_df[['Name', 'Status']].copy()
filtered_names['Normalized Name'] = filtered_names['Name'].str.lower().str.strip()

# Initialize lists for different message categories
active_messages = []  # Active and exists in LIA
not_in_lia_messages = []  # Active in EG2 but not in LIA
terminated_in_lia_messages = []  # Terminated but still in LIA
doesnt_exist_in_caar_messages = []  # In LIA but not in CAAR
doesnt_have_status_messages = []  # No status in CAAR but in LIA

# Check CAAR EG2 records
for _, row in filtered_names.iterrows():
    if row['Name'].strip() == '' or row['Status'].strip() == '':
        continue

    name, status = row['Name'].strip(), row['Status'].strip()
    normalized_name = row['Normalized Name']

    if status == 'Active' and normalized_name in radgrid_names.values:
        active_messages.append(f"{name} is active in EG2 and has access in LIA")
    elif status == 'Active' and normalized_name not in radgrid_names.values:
        not_in_lia_messages.append(f"{name}'s status is active in CAAR EG2, but doesn't exist in LIA. Please confirm {name} access in LIA immediately")
    elif status == 'Terminated' and normalized_name in radgrid_names.values:
        terminated_in_lia_messages.append(f"{name} has been terminated from CAAR EG2. Please remove from LIA")

# Check LIA records
for name in radgrid_df['Credential Owner']:
    if pd.isna(name) or name.strip() == '':
        continue

    normalized_name = name.lower().strip()
    matching_names = filtered_names[filtered_names['Normalized Name'] == normalized_name]

    if matching_names.empty:
        doesnt_exist_in_caar_messages.append(f"{name} does not exist in CAAR EG2. Not a customer")
    else:
        status_rows = matching_names[
            (matching_names['Status'].isna()) |
            (matching_names['Status'].str.strip() == '')
        ]
        if not status_rows.empty:
            doesnt_have_status_messages.append(f"{name} does not have a status in CAAR EDGE 2. Please update the status immediately")

# Create a DataFrame with the results
output_df = pd.DataFrame({
    'Active': pd.Series(active_messages),
    'Not in LIA': pd.Series(not_in_lia_messages),
    "Terminated in LIA": pd.Series(terminated_in_lia_messages),
    "Doesn't exist in CAAR EDGE 2 (Probably not a customer)": pd.Series(doesnt_exist_in_caar_messages),
    "Doesn't have status in CAAR EDGE 2, but exist in LIA": pd.Series(doesnt_have_status_messages)
})

# Save to CSV
output_df.to_csv(output_csv_file, index=False)

# Print summary
print(f"Results have been saved to {output_csv_file}")
total_issues = sum([
    len(active_messages),
    len(not_in_lia_messages),
    len(terminated_in_lia_messages),
    len(doesnt_exist_in_caar_messages),
    len(doesnt_have_status_messages)
])
print(f"Total number of issues found: {total_issues}")

In [ ]:
# Combine the data into one dataframe. Then simply check for the status in the CAAR vs KMS. For every existing user in the CAAR shouldnt have administrator terminal access level.
# Load filtered EG1 and EG2 DataFrames
filtered_EG1_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\filtered_output_EG1.csv'
filtered_EG2_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG2_CAAR_Filtered.csv'

output_EG1_df = pd.read_csv(filtered_EG1_path)
output_EG2_df = pd.read_csv(filtered_EG2_path)

# Combine the DataFrames
combined_EG_df = pd.concat([output_EG1_df, output_EG2_df], ignore_index=True)

# Save the combined result to a new CSV file
combined_EG_df.to_csv(r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG12_Filtered_Combined.csv", index=False)

# Display the result
print(combined_EG_df.head())  # Print the first few rows of the combined result
print(f"Total count of combined names: {len(combined_EG_df)}")

# Load the datasets
kms_file_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_user_data_final.csv'
combined_caar_file_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG12_Filtered_Combined.csv'

# Read the CSV files
kms_data = pd.read_csv(kms_file_path)
combined_caar_data = pd.read_csv(combined_caar_file_path)

# Combine 'First Name' and 'Last Name' in KMS data to create a full name column
kms_data['Full Name'] = kms_data['First Name'] + ' ' + kms_data['Last Name']

# Prepare a list to store the results
results = []

# Loop through each name in KMS data and perform the checks
for _, row in kms_data.iterrows():
    full_name = row['Full Name']
    terminal_level = row['Terminal user level']

    # Check if the name exists in the COMBINED dataset
    if full_name in combined_caar_data['Name'].values:
        # Check the terminal user level
        if terminal_level == "User":
            result = f"{full_name} has a correct terminal user level."
        else:
            result = f"{full_name} has an incorrect terminal access level."
    else:
        result = f"{full_name} is not found in the CAAR EG1 & EG2, probably from EDGE DC."

    # Append the result to the list
    results.append({"Full Name": full_name, "Result": result})

# Convert the results to a DataFrame
results_df = pd.DataFrame(results)

# Save the results to a CSV file
output_file_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_Terminal_Access_Validated.csv'
results_df.to_csv(output_file_path, index=False)

print(f"Results have been saved to {output_file_path}")

# **Checking Users' status on all KMS user at both EDGE 1 and EDGE 2**

In [ ]:
# Remove pending and unknown status coz useless
# Checking KMS User data at EG1
kms_data_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_user_data_final.csv'
filtered_output_path_EG1 = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\filtered_output_EG1.csv'

# Read the CSV files
kms_df = pd.read_csv(kms_data_path)
filtered_df_EG1 = pd.read_csv(filtered_output_path_EG1)

# Combine 'First Name' and 'Last Name' in the KMS data to create a 'Full Name' column
kms_df['Full Name'] = kms_df['First Name'] + ' ' + kms_df['Last Name']

# Extract only the relevant columns from each dataframe
kms_names = kms_df['Full Name'].unique()
filtered_names_EG1 = filtered_df_EG1[['Name', 'Status']].dropna(subset=['Name'])

# Identify common names between the two datasets
common_names_EG1 = set(kms_names).intersection(set(filtered_names_EG1['Name']))

# Filter the data for only the common names in the filtered output dataset
common_names_df_EG1 = filtered_names_EG1[filtered_names_EG1['Name'].isin(common_names_EG1)]

# Separate common names by their status
active_users_EG1 = []
terminated_users_EG1 = []

# Classify each common name based on its status and add to corresponding list
for _, row in common_names_df_EG1.iterrows():
    name, status = row['Name'], row['Status']
    if status == "Active":
        active_users_EG1.append(name)
    elif status == "Terminate":
        terminated_users_EG1.append(name)

# Convert each list into a separate DataFrame
active_df_EG1 = pd.DataFrame(active_users_EG1, columns=['Name'])
terminated_df_EG1 = pd.DataFrame(terminated_users_EG1, columns=['Name'])

# Add a Status column to each DataFrame
active_df_EG1['Status'] = 'The status is Active, thus is ok to exist in KMS'
terminated_df_EG1['Status'] = 'The status is terminated, thus not ok to exist in KMS'

# Concatenate all status groups into one DataFrame
all_status_df_EG1 = pd.concat([active_df_EG1, terminated_df_EG1], ignore_index=True)

# Save to CSV
output_path_EG1 = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_user_status_EG1.csv'
all_status_df_EG1.to_csv(output_path_EG1, index=False)

print(f"CSV file with grouped statuses saved at: {output_path_EG1}")

In [ ]:
import pandas as pd

# Checking KMS User data at EG2
kms_data_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_user_data_final.csv'
filtered_output_path_EG2 = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG2_CAAR_Filtered.csv'

# Read the CSV files
kms_df = pd.read_csv(kms_data_path)
filtered_df_EG2 = pd.read_csv(filtered_output_path_EG2)

# Combine 'First Name' and 'Last Name' in the KMS data to create a 'Full Name' column
kms_df['Full Name'] = kms_df['First Name'] + ' ' + kms_df['Last Name']

# Extract only the relevant columns from each dataframe
kms_names = kms_df['Full Name'].unique()
filtered_names_EG2 = filtered_df_EG2[['Name', 'Status']].dropna(subset=['Name'])

# Identify common names between the two datasets
common_names_EG2 = set(kms_names).intersection(set(filtered_names_EG2['Name']))

# Filter the data for only the common names in the filtered output dataset
common_names_df_EG2 = filtered_names_EG2[filtered_names_EG2['Name'].isin(common_names_EG2)]

# Separate common names by their status
active_users_EG2 = []
pending_users_EG2 = []
terminated_users_EG2 = []
unknown_status_users_EG2 = []

# Classify each common name based on its status and add to corresponding list
for _, row in common_names_df_EG2.iterrows():
    name, status = row['Name'], row['Status']
    if status == "Active":
        active_users_EG2.append(name)
    elif status == "Terminate":
        terminated_users_EG2.append(name)

# Convert each list into a separate DataFrame
active_df_EG2 = pd.DataFrame(active_users_EG2, columns=['Name'])
terminated_df_EG2 = pd.DataFrame(terminated_users_EG2, columns=['Name'])

# Add a Status column to each DataFrame
active_df_EG2['Status'] = 'Active'
terminated_df_EG2['Status'] = 'Terminate'

# Concatenate all status groups into one DataFrame
all_status_df_EG2 = pd.concat([active_df_EG2, terminated_df_EG2], ignore_index=True)

# Save to CSV
output_path_EG2 = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_user_status_EG2.csv'
all_status_df_EG2.to_csv(output_path_EG2, index=False)

print(f"CSV file with grouped statuses saved at: {output_path_EG2}")

# **Checking Users' RFID Number across all platform LIA (EDGE 2), P1 (EDGE 1), and KMS (Both EDGE 1 and EDGE 2)**

In [ ]:
import pandas as pd
import re

# Combine 'First Name' and 'Last Name' in the KMS data to create a 'Full Name' column
KMS_Name_RFID['Full Name'] = KMS_Name_RFID['First Name'] + ' ' + KMS_Name_RFID['Last Name']

# Extract relevant columns from each dataframe
kms_names_total = KMS_Name_RFID[['Full Name', 'RFID']]
P1_names_RFID = P1_user_data_RFID[['Staff Name', 'Card No']].dropna(subset=['Staff Name', 'Card No'])
LIA_names_RFID = LIA_name_RFID_df[['Credential Owner', 'Credential']].dropna(subset=['Credential Owner', 'Credential'])

def normalize_name(name):
    """Normalize name by converting to lowercase and removing special characters."""
    name = re.sub(r'[^a-zA-Z\s]', '', name)  # Remove non-alphabet characters
    return " ".join(name.lower().strip().split())  # Convert to lowercase and strip extra spaces

def get_name_components(name):
    """Split the full name into a list of components (first, middle, last)."""
    return normalize_name(name).split()

def count_matching_components(name1, name2):
    """Count matching components between two names."""
    components1 = get_name_components(name1)
    components2 = get_name_components(name2)

    # Find the number of common components
    matching_components = len(set(components1) & set(components2))

    return matching_components

def are_names_identical(name1, name2):
    """Determine if two names are identical based on matching components."""
    # Consider names identical if they have at least two matching components
    return count_matching_components(name1, name2) >= 2

def check_rfid_consistency_by_component_matching(LIA_names_RFID, P1_names_RFID, kms_names_total, output_file='rfid_consistency_results.csv'):
    # Convert names to strings and create dictionaries
    lia_dict = dict(zip(LIA_names_RFID.iloc[:, 0].astype(str), LIA_names_RFID.iloc[:, 1]))
    p1_dict = dict(zip(P1_names_RFID.iloc[:, 0].astype(str), P1_names_RFID.iloc[:, 1]))
    kms_dict = dict(zip(kms_names_total.iloc[:, 0].astype(str), kms_names_total.iloc[:, 1]))

    # Gather all unique names
    all_names = list(lia_dict.keys()) + list(p1_dict.keys()) + list(kms_dict.keys())
    unique_names = []

    # Filter out duplicate names based on component matching
    for name in all_names:
        if not any(are_names_identical(name, existing_name) for existing_name in unique_names):
            unique_names.append(name)

    # Initialize lists for results
    results = []

    # Process each unique name
    for name in unique_names:
        present_in = []
        for df_name, data_dict in [('LIA', lia_dict), ('P1', p1_dict), ('KMS', kms_dict)]:
            # Find a matching name based on component comparison
            match_name = next((key for key in data_dict if are_names_identical(key, name)), None)
            if match_name:
                present_in.append((df_name, data_dict[match_name]))

        # Case 1: Single dataframe
        if len(present_in) == 1:
            results.append({
                'Name': name,
                'Category': 'Single Dataframe',
                'Dataframe': present_in[0][0],
                'RFID': present_in[0][1]
            })
            continue

        # Case 2: Check identical RFID across all three
        rfid_values = set(str(entry[1]) for entry in present_in)
        if len(present_in) == 3 and len(rfid_values) == 1:
            results.append({
                'Name': name,
                'Category': 'Identical RFID',
                'Dataframes': ', '.join(df for df, _ in present_in),
                'RFID': list(rfid_values)[0]
            })
            continue

        # Case 3: Partial matches
        rfid_matches = [(present_in[i][0], present_in[j][0])
                        for i in range(len(present_in))
                        for j in range(i + 1, len(present_in))
                        if str(present_in[i][1]) == str(present_in[j][1])]
        if rfid_matches:
            missing_dfs = set(['LIA', 'P1', 'KMS']) - set(df for pair in rfid_matches for df in pair)
            results.append({
                'Name': name,
                'Category': 'Partial RFID Match',
                'Matching Dataframes': ' and '.join([f'{pair[0]}-{pair[1]}' for pair in rfid_matches]),
                'Different Dataframes': ', '.join(missing_dfs)
            })

    # Convert results to DataFrame and save to CSV
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_file, index=False)

    # Print confirmation message
    print(f"\nRFID consistency results saved to {output_file}")
    print(f"Total entries processed: {len(results)}")

    return results_df

# Example usage
results = check_rfid_consistency_by_component_matching(LIA_names_RFID, P1_names_RFID, kms_names_total, r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\RFID_Checked.csv')

# **Checking Users' KeyTag Access on KMS at both EDGE 1 and EDGE 2**

In [ ]:
import pandas as pd
import json

### EG1 Code
def get_keytag_eg1(row, sub_row, col):
    if sub_row == 1:
        return f"C{row:02d}{col:02d}"
    elif sub_row == 2:
        adjusted_col = col + 16
        return f"C{row:02d}{adjusted_col:02d}"
    return None

def map_keytags_to_groups_eg1(df_eg1):
    keytag_mapping_eg1 = {}
    current_row_eg1 = None

    for idx, row in df_eg1.iterrows():
        if pd.isna(row['Final']):
            continue

        physical_row_eg1 = row['Final']

        if current_row_eg1 != physical_row_eg1:
            current_row_eg1 = physical_row_eg1
            sub_row_eg1 = 1
        else:
            sub_row_eg1 = 2

        for col in range(1, 17):
            col_label = f"{col:02d}"
            keytag_group_eg1 = row[col_label]

            if pd.notna(keytag_group_eg1):
                keytag_eg1 = get_keytag_eg1(int(physical_row_eg1), sub_row_eg1, col)

                if keytag_group_eg1 not in keytag_mapping_eg1:
                    keytag_mapping_eg1[keytag_group_eg1] = []
                if keytag_eg1 and keytag_eg1 not in keytag_mapping_eg1[keytag_group_eg1]:
                    keytag_mapping_eg1[keytag_group_eg1].append(keytag_eg1)

    return keytag_mapping_eg1

def save_mapping_as_csv_eg1(mapping_eg1, output_path_eg1):
    rows_eg1 = []
    for group in sorted(mapping_eg1.keys()):
        sorted_keytags_eg1 = sorted(mapping_eg1[group])
        rows_eg1.append({
            "KeyTag Group": group,
            "KeyTags": ", ".join(sorted_keytags_eg1)
        })

    df_eg1 = pd.DataFrame(rows_eg1)
    df_eg1.to_csv(output_path_eg1, index=False)
    return df_eg1

def process_key_rack_mapping_eg1(input_file_path_eg1, output_file_path_eg1):
    try:
        excel_data_eg1 = pd.ExcelFile(input_file_path_eg1)
        key_rack_access_df_eg1 = excel_data_eg1.parse('Key-Rack-Access')
        keytag_mapping_eg1 = map_keytags_to_groups_eg1(key_rack_access_df_eg1)
        keytag_df_eg1 = save_mapping_as_csv_eg1(keytag_mapping_eg1, output_file_path_eg1)

        print(f"\nEG1 Mapping created successfully!")
        print(f"Results saved to: {output_file_path_eg1}")
        print("\nEG1 Mapping preview:")
        print(keytag_df_eg1)

        return keytag_df_eg1

    except Exception as e:
        print(f"Error processing EG1 file: {str(e)}")
        return None

### EG2 Code
def get_keytag_eg2(row, sub_row, col):
    if sub_row == 1:
        return f"2C{row:02d}{col:02d}"
    elif sub_row == 2:
        adjusted_col = col + 16
        return f"2C{row:02d}{adjusted_col:02d}"
    return None

def map_keytags_to_groups_eg2(df_eg2):
    keytag_mapping_eg2 = {}
    current_row_eg2 = None

    for idx, row in df_eg2.iterrows():
        if pd.isna(row['Final']):
            continue

        physical_row_eg2 = row['Final']

        if current_row_eg2 != physical_row_eg2:
            current_row_eg2 = physical_row_eg2
            sub_row_eg2 = 1
        else:
            sub_row_eg2 = 2

        for col in range(1, 17):
            col_label = f"{col:02d}"
            keytag_group_eg2 = row[col_label]

            if pd.notna(keytag_group_eg2):
                keytag_eg2 = get_keytag_eg2(int(physical_row_eg2), sub_row_eg2, col)

                if keytag_group_eg2 not in keytag_mapping_eg2:
                    keytag_mapping_eg2[keytag_group_eg2] = []
                if keytag_eg2 and keytag_eg2 not in keytag_mapping_eg2[keytag_group_eg2]:
                    keytag_mapping_eg2[keytag_group_eg2].append(keytag_eg2)

    return keytag_mapping_eg2

def save_mapping_as_csv_eg2(mapping_eg2, output_path_eg2):
    rows_eg2 = []
    for group in sorted(mapping_eg2.keys()):
        sorted_keytags_eg2 = sorted(mapping_eg2[group])
        rows_eg2.append({
            "KeyTag Group": group,
            "KeyTags": ", ".join(sorted_keytags_eg2)
        })

    df_eg2 = pd.DataFrame(rows_eg2)
    df_eg2.to_csv(output_path_eg2, index=False)
    return df_eg2

def process_key_rack_mapping_eg2(input_file_path_eg2, output_file_path_eg2):
    try:
        excel_data_eg2 = pd.ExcelFile(input_file_path_eg2)
        key_rack_access_df_eg2 = excel_data_eg2.parse('Key-Rack-Access')
        keytag_mapping_eg2 = map_keytags_to_groups_eg2(key_rack_access_df_eg2)
        keytag_df_eg2 = save_mapping_as_csv_eg2(keytag_mapping_eg2, output_file_path_eg2)

        print(f"\nEG2 Mapping created successfully!")
        print(f"Results saved to: {output_file_path_eg2}")
        print("\nEG2 Mapping preview:")
        print(keytag_df_eg2)

        return keytag_df_eg2

    except Exception as e:
        print(f"Error processing EG2 file: {str(e)}")
        return None

# Main execution and validation
def main():
    # File paths for EG1
    input_file_path_eg1 = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\Cust-Master-Data (3).xlsx'
    output_csv_path_eg1 = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG1_RackMapping_new.csv'

    # File paths for EG2
    input_file_path_eg2 = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\Cust-Master-Data_EDGE2 (3).xlsx'
    output_csv_path_eg2 = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\EG2_RackMapping_new.csv'

    # Process EG1 and EG2
    result_df_eg1 = process_key_rack_mapping_eg1(input_file_path_eg1, output_csv_path_eg1)
    result_df_eg2 = process_key_rack_mapping_eg2(input_file_path_eg2, output_csv_path_eg2)

    # Merge KeyTag mappings from EG1 and EG2
    merged_keytag_mapping = {}

    # Merge EG1 mappings
    for _, row in result_df_eg1.iterrows():
        group = row['KeyTag Group']
        keytags = row['KeyTags'].split(', ')

        if group not in merged_keytag_mapping:
            merged_keytag_mapping[group] = set()
        merged_keytag_mapping[group].update(keytags)

    # Merge EG2 mappings
    for _, row in result_df_eg2.iterrows():
        group = row['KeyTag Group']
        keytags = row['KeyTags'].split(', ')

        if group not in merged_keytag_mapping:
            merged_keytag_mapping[group] = set()
        merged_keytag_mapping[group].update(keytags)

    # Create merged DataFrame
    merged_rows = []
    for group, keytags in merged_keytag_mapping.items():
        merged_rows.append({
            "KeyTag Group": group,
            "KeyTags": ", ".join(sorted(keytags))
        })

    combined_keytag_df = pd.DataFrame(merged_rows)

    # Save merged mapping
    combined_keytag_output_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\Key_Rack_Combined.csv'
    combined_keytag_df.to_csv(combined_keytag_output_path, index=False)
    print(f"\nMerged KeyTag Mapping saved to: {combined_keytag_output_path}")

    # Create mapping of Cust groups to allowed KeyTags
    cust_group_to_keytags = {
        row["KeyTag Group"]: set(row["KeyTags"].split(", "))
        for _, row in combined_keytag_df.iterrows()
    }

    # Load user data
    user_data_file_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_user_data_final.csv'
    user_data_df = pd.read_csv(user_data_file_path)

    # Filter out Administration users and keep only User level users
    user_data_df = user_data_df[user_data_df['Terminal user level'] == 'User']

    # Validation results
    validation_results = []

    # Process each user
    for _, user in user_data_df.iterrows():
        # Extract user details
        user_name = f"{user['First Name']} {user['Last Name']}"

        # Split Cust groups and KeyTags
        cust_groups = str(user["Cust group"]).split(" ;")
        key_tags = set(str(user["Key tag"]).replace("keyTag ", "").split(" ;"))

        # Collect all valid KeyTags based on user's Cust groups
        valid_keytags = set()
        for group in cust_groups:
            valid_keytags.update(cust_group_to_keytags.get(group, set()))

        # Determine valid and invalid KeyTags
        valid_user_keytags = key_tags.intersection(valid_keytags)
        invalid_user_keytags = key_tags - valid_keytags

        # Record result
        validation_results.append({
            'User Name': user_name,
            'Terminal User Level': user['Terminal user level'],  # Added Terminal User Level
            'Cust Group': " ; ".join(cust_groups),
            'Valid KeyTags': ", ".join(valid_user_keytags) if valid_user_keytags else "None",
            'Invalid KeyTags': ", ".join(invalid_user_keytags) if invalid_user_keytags else "None"
        })

    # Create DataFrame from validation results
    validation_df = pd.DataFrame(validation_results)

    # Save validation results
    validation_output_path = r'C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_User_Data_Validated.csv'
    validation_df.to_csv(validation_output_path, index=False)
    print(f"\nKeyTag Access Validation results saved to: {validation_output_path}")

    # Optional: Print results to console
    print("\nValidation Results Preview:")
    print(validation_df)

if __name__ == "__main__":
    main()

# **Final result**

In [ ]:
import pandas as pd
import os  # For file name handling

# List of CSV files to combine
csv_files = [
    r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\P1_Credential_Checked.csv",
    r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\LIA_Credentials_Checked.csv",
    r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_Terminal_Access_Validated.csv",
    r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_user_status_EG1.csv",
    r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_user_status_EG2.csv",
    r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\KMS_User_Data_Validated.csv",
    r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\RFID_Checked.csv"
]

# Output Excel file path
output_excel_file = r"C:\Users\adonis.hugo\Downloads\Access_Reviewer_Run_May_2025\Automatic_Access_Reviewer_V3_Local.xlsx"

# Create an Excel writer object
with pd.ExcelWriter(output_excel_file, engine="openpyxl") as writer:
    for csv_file in csv_files:
        try:
            # Read the CSV file
            df = pd.read_csv(csv_file)
            # Generate a safe sheet name from the file name (without extension)
            sheet_name = os.path.basename(csv_file).split(".")[0]  # Extract base file name
            # Replace invalid characters in the sheet name
            sheet_name = sheet_name.replace(":", "_").replace("/", "_").replace("\\", "_")
            # Write DataFrame to a sheet
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)  # Excel sheet names max 31 chars
        except Exception as e:
            print(f"Error processing file {csv_file}: {e}")

print(f"Combined Excel file saved to {output_excel_file}")
